In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv('train.csv')
ids = sorted(df['image_id'].astype(str).unique())[:1200]
sub = df[df['image_id'].astype(str).isin(ids)].copy()

def decode(rle, h=128, w=800):
    m = np.zeros(h * w, dtype=np.uint8)
    if isinstance(rle, str) and rle.strip():
        s = list(map(int, rle.split()))
        for st, ln in zip(s[0::2], s[1::2]):
            a = st - 1
            b = min(a + ln, h * w)
            if a < h * w:
                m[a:b] = 1
    return m.reshape((h, w), order='F')

sub['pos'] = sub['mask_rle'].apply(lambda r: decode(r).sum() > 0)
classes = ['defect_a', 'defect_b', 'defect_c', 'defect_d']
pv = sub.pivot_table(index='image_id', columns='defect_type', values='pos', aggfunc='max').reindex(ids)[classes].fillna(False).astype(bool)

def combo(row):
    p = [c for c in classes if row[c]]
    return '+'.join(p) if p else 'clean'

pv['combo'] = pv.apply(combo, axis=1)
counts = pv['combo'].value_counts()

a = int((pv['combo'] == 'clean').sum())
b = [int(pv[c].sum()) for c in classes]
c = int(counts.shape[0])
d = (int(counts.max()), int(counts.min()))

print(len(ids), len(sub))
print(counts)
print(f"{a}; {b[0]}, {b[1]}, {b[2]}, {b[3]}; {c}; {d[0]}, {d[1]}")

1200 4800
combo
clean                         578
defect_c                      334
defect_d                       81
defect_c+defect_d              71
defect_a                       59
defect_b                       22
defect_a+defect_c              21
defect_a+defect_d              14
defect_a+defect_c+defect_d      7
defect_b+defect_d               3
defect_a+defect_b+defect_d      3
defect_a+defect_b               2
defect_b+defect_c+defect_d      2
defect_b+defect_c               2
defect_a+defect_b+defect_c      1
Name: count, dtype: int64
578; 107, 35, 438, 181; 15; 578, 1


In [3]:
bsub = sub[sub['defect_type'] == 'defect_b']
areas = np.array([int(decode(r).sum()) for r in bsub['mask_rle']])

total = 1200 * 128 * 800
pos = int(areas.sum())
neg = total - pos

a = pos / total * 100
b = areas[areas > 0].mean()
c = round(neg / pos)

print(len(bsub), pos, int((areas > 0).sum()))
print(f"{a:.4f}, {b:.1f}, {c}")

1200 19654 35
0.0160, 561.5, 6251


In [4]:
n_clean = counts['clean']
gw = np.minimum(np.sqrt(n_clean / counts), 5.0)
w = pv['combo'].map(gw)
p = w / w.sum()
is_b = pv['combo'].str.contains('defect_b')

a_s = p[pv['combo'] == 'clean'].sum() * 100
a_n = (pv['combo'] == 'clean').mean() * 100
b_s = p[is_b].sum() * 1200
b_n = float(is_b.sum())
c = b_s / b_n
d = int((np.sqrt(n_clean / counts) >= 5).sum())

print(gw)
print(f"{a_s:.2f}%, {a_n:.2f}%; {b_s:.2f}, {b_n:.2f}; {c:.2f}x; {d}")

combo
clean                         1.000000
defect_c                      1.315499
defect_d                      2.671292
defect_c+defect_d             2.853217
defect_a                      3.129954
defect_b                      5.000000
defect_a+defect_c             5.000000
defect_a+defect_d             5.000000
defect_a+defect_c+defect_d    5.000000
defect_b+defect_d             5.000000
defect_a+defect_b+defect_d    5.000000
defect_a+defect_b             5.000000
defect_b+defect_c+defect_d    5.000000
defect_b+defect_c             5.000000
defect_a+defect_b+defect_c    5.000000
Name: count, dtype: float64
28.81%, 48.17%; 104.69, 35.00; 2.99x; 10


In [5]:
import math

base = 3e-4

def m(t):
    if t < 60:
        return t / 60
    return 0.5 * (1 + math.cos(math.pi * (t - 60) / 420))

for t in [15, 270, 400]:
    print(t, f"{base * m(t):.2e}")

print("bug", f"{base * m(8):.2e}")

15 7.50e-05
270 1.50e-04
400 2.61e-05
bug 4.00e-05


In [6]:
scores = [0.512, 0.601, 0.648, 0.661, 0.664, 0.668, 0.671, 0.670, 0.672, 0.669, 0.673, 0.665]

def trace(scores, patience, min_delta):
    best = float('-inf')
    best_ep = None
    wait = 0
    for ep, s in enumerate(scores, start=1):
        if s > best + min_delta + 1e-12:
            best, best_ep, wait = s, ep, 0
        else:
            wait += 1
        print(ep, s, best_ep, round(best, 3), wait)
        if wait >= patience:
            return ep, best_ep, best
    return None, best_ep, best

print(trace(scores, 3, 0.002))
print(trace(scores, 3, 0.0))

1 0.512 1 0.512 0
2 0.601 2 0.601 0
3 0.648 3 0.648 0
4 0.661 4 0.661 0
5 0.664 5 0.664 0
6 0.668 6 0.668 0
7 0.671 7 0.671 0
8 0.67 7 0.671 1
9 0.672 7 0.671 2
10 0.669 7 0.671 3
(10, 7, 0.671)
1 0.512 1 0.512 0
2 0.601 2 0.601 0
3 0.648 3 0.648 0
4 0.661 4 0.661 0
5 0.664 5 0.664 0
6 0.668 6 0.668 0
7 0.671 7 0.671 0
8 0.67 7 0.671 1
9 0.672 9 0.672 0
10 0.669 9 0.672 1
11 0.673 11 0.673 0
12 0.665 11 0.673 1
(None, 11, 0.673)


In [7]:
params = 25_000_000
static_mb = params * 4 * 4 / 1e6

act_fp32 = 150
act_amp = 75

b_fp32 = static_mb + 16 * act_fp32
b_amp = static_mb + 16 * act_amp

budget = 3400
c_fp32 = int((budget - static_mb) // act_fp32)
c_amp = int((budget - static_mb) // act_amp)

micro = 8
accum = 4
n_train = 960

d = micro * accum
micro_batches = -(-n_train // micro)
steps = -(-micro_batches // accum)
f = accum
g = 2 * steps

print(n_train % micro, micro_batches % accum)
print(f"{static_mb:.0f}; {b_fp32:.0f}, {b_amp:.0f}; {c_fp32}, {c_amp}; {d}; {micro_batches}, {steps}; {f}; {g}")

0 0
400; 2800, 1600; 20, 40; 32; 120, 30; 4; 60


In [8]:
sub['area'] = sub['mask_rle'].apply(lambda r: int(decode(r).sum()))
gt_empty = sub['area'] == 0
n_pairs = len(sub)

a = int(gt_empty.sum())
b = a / n_pairs
fg_dice_base = 0.0
fp_score_base = 1.0
empty_acc_base = 1.0
c = 0.70 * fg_dice_base + 0.20 * fp_score_base + 0.10 * empty_acc_base

n_pos, n_empty = 60, 340
fg_dice = 0.50
n_clean_pred, n_blob = 306, 34
blob = 1024
canvas = 102400

fp_score = 1 - (n_blob * blob / canvas) / n_empty
empty_acc = n_clean_pred / n_empty
d_official = 0.70 * fg_dice + 0.20 * fp_score + 0.10 * empty_acc
d_naive = (n_pos * fg_dice + n_clean_pred * 1 + n_blob * 0) / (n_pos + n_empty)

base_naive = (n_pos * 0 + n_empty * 1) / (n_pos + n_empty)
base_official = 0.70 * 0 + 0.20 * 1 + 0.10 * 1

print(n_pairs, a)
print(base_official, base_naive)
print(f"{a}; {b:.4f}, {c:.4f}; {d_official:.4f}, {d_naive:.4f}; naive Dice, {base_naive:.4f}")

4800 4039
0.30000000000000004 0.85
4039; 0.8415, 0.3000; 0.6398, 0.8400; naive Dice, 0.8500
